# Clustering Algorithm Comparison
## Machine learning for energy applications — Module 3, Exercise 5

In this exercise, we will compare four clustering algorithms on the same well log dataset:
1. **K-Means**
2. **DBSCAN**
3. **Hierarchical (Agglomerative) Clustering**
4. **Gaussian Mixture Models (GMM)**

Each algorithm has different strengths and assumptions. By applying all four to the same data and comparing results, we can better understand when each method is most appropriate.

**Topics covered:**
1. Load and explore the dataset
2. Standardize features
3. Apply K-Means (K=3)
4. Apply DBSCAN
5. Apply Hierarchical Clustering
6. Apply GMM
7. Side-by-side visualization
8. Silhouette score comparison
9. Cluster assignment comparison
10. Summary table and discussion

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['figure.dpi'] = 100

---
## 1. Load and Explore the Dataset

Load `w5.csv`, which contains well log measurements, and inspect its structure.

In [ ]:
# Load the dataset
df = pd.read_csv('w5.csv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nSummary statistics:")
df.describe()

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Pairplot to visualize relationships between features
sns.pairplot(df.dropna(), diag_kind='kde', plot_kws={'alpha': 0.4, 's': 15})
plt.suptitle('Pairplot of Well Log Features', y=1.02, fontsize=14)
plt.show()

---
## 2. Standardize Features

Drop any rows with missing values, then standardize the numeric features: the same six logs as in Lab 1 (DEPTH excluded).

**Exploratory analysis of one fixed dataset.** The scaler is fitted on all samples of W5 because we only describe this well; nothing is predicted for new data, so there is no training/validation/test split.

In [ ]:
# Drop missing values and select numeric features
df_clean = df.dropna().copy()
feature_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()

# Exclude DEPTH if present: it records where a sample is, not what it is
if 'DEPTH' in feature_cols:
    feature_cols.remove('DEPTH')

print(f"Features for clustering: {feature_cols}")
print(f"Number of samples after dropping NaN: {len(df_clean)}")

X = df_clean[feature_cols].values

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nScaled data shape: {X_scaled.shape}")
print(f"Mean: {X_scaled.mean(axis=0).round(4)}")
print(f"Std:  {X_scaled.std(axis=0).round(4)}")

---
## 3. Apply K-Means (K=3)

Fit K-Means with 3 clusters. K = 3 is carried over from the silhouette analysis in Lab 1 (highest mean silhouette for K = 2–6 on these six standardized logs), and we use the same K for Ward and the mixture so that the partitions can be compared. It is a working choice, not a discovered number of facies.

In [ ]:
# K-Means
km = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_km = km.fit_predict(X_scaled)

print("K-Means cluster sizes:")
unique, counts = np.unique(labels_km, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Cluster {u}: {c} samples")

print(f"\nInertia: {km.inertia_:.2f}")

---
## 4. Apply DBSCAN

DBSCAN identifies dense regions and marks sparse points as noise (-1). We need to choose `eps` and `min_samples`. As in Lab 2, `min_samples = 5` and `eps` is read from the k-distance plot, where the sorted distances start to rise steeply.

In [ ]:
# DBSCAN — use a k-distance plot to help choose eps
from sklearn.neighbors import NearestNeighbors

k = 5
nn = NearestNeighbors(n_neighbors=k)
nn.fit(X_scaled)
distances, _ = nn.kneighbors(X_scaled)
k_distances = np.sort(distances[:, -1])

# eps read from the plot: the curve starts to rise steeply at about 0.5
# (close to the 90th percentile of the 5-distances), as in Lab 2
eps_val = 0.5

plt.figure(figsize=(10, 5))
plt.plot(k_distances)
plt.axhline(eps_val, color='tomato', linestyle='--', label=f'chosen eps = {eps_val}')
plt.legend()
plt.xlabel('Points (sorted by distance)')
plt.ylabel(f'{k}-NN Distance')
plt.title(f'{k}-Distance Plot for DBSCAN eps Selection')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Median {k}-distance: {np.median(k_distances):.2f}; "
      f"90th percentile: {np.percentile(k_distances, 90):.2f}")
print(f"Share of samples with {k}-distance <= eps: {np.mean(k_distances <= eps_val):.1%}")

In [ ]:
# Apply DBSCAN
# eps_val was read from the k-distance plot above
min_samples_val = 5

db = DBSCAN(eps=eps_val, min_samples=min_samples_val)
labels_db = db.fit_predict(X_scaled)

n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = (labels_db == -1).sum()

print(f"DBSCAN results (eps={eps_val}, min_samples={min_samples_val}):")
print(f"  Number of clusters: {n_clusters_db}")
print(f"  Noise points: {n_noise}")
print(f"\nCluster sizes:")
unique, counts = np.unique(labels_db, return_counts=True)
for u, c in zip(unique, counts):
    label_name = 'Noise' if u == -1 else f'Cluster {u}'
    print(f"  {label_name}: {c} samples")

---
## 5. Apply Hierarchical Clustering

In [ ]:
# Hierarchical Clustering (Ward linkage, K=3)
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_agg = agg.fit_predict(X_scaled)

print("Hierarchical (Ward) cluster sizes:")
unique, counts = np.unique(labels_agg, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Cluster {u}: {c} samples")

---
## 6. Apply Gaussian Mixture Model

Three components, for comparability with K-Means and Ward. To choose the number of components for the mixture itself, compare BIC over K as in Lab 3.

In [ ]:
# GMM with 3 components
gmm = GaussianMixture(n_components=3, random_state=42, n_init=5)
gmm.fit(X_scaled)
labels_gmm = gmm.predict(X_scaled)

print("GMM cluster sizes:")
unique, counts = np.unique(labels_gmm, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Cluster {u}: {c} samples")

print(f"\nCluster weights: {gmm.weights_.round(4)}")
print(f"BIC: {gmm.bic(X_scaled):.2f}")

---
## 7. Side-by-Side Visualization (2x2 Grid)

Compare all four clustering results using the first two features (or first two PCA components if there are more than 2 features).

In [ ]:
# If more than 2 features, use PCA for visualization
from sklearn.decomposition import PCA

if X_scaled.shape[1] > 2:
    pca = PCA(n_components=2)
    X_vis = pca.fit_transform(X_scaled)
    xlabel = f'PC1 ({pca.explained_variance_ratio_[0]:.1%})'
    ylabel = f'PC2 ({pca.explained_variance_ratio_[1]:.1%})'
    print(f"Using PCA for visualization. Total variance explained: {pca.explained_variance_ratio_.sum():.1%}")
else:
    X_vis = X_scaled
    xlabel = feature_cols[0] + ' (scaled)'
    ylabel = feature_cols[1] + ' (scaled)'

In [ ]:
# 2x2 comparison plot
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

methods = ['K-Means (K=3)', 'DBSCAN', 'Hierarchical (Ward, K=3)', 'GMM (K=3)']
all_labels = [labels_km, labels_db, labels_agg, labels_gmm]

for ax, name, labels in zip(axes.ravel(), methods, all_labels):
    # Use a different colour for noise points in DBSCAN
    if -1 in labels:
        noise_mask = labels == -1
        ax.scatter(X_vis[noise_mask, 0], X_vis[noise_mask, 1], c='gray',
                   marker='x', s=30, alpha=0.5, label='Noise')
        ax.scatter(X_vis[~noise_mask, 0], X_vis[~noise_mask, 1], c=labels[~noise_mask],
                   cmap='viridis', alpha=0.6, s=30, edgecolors='k', linewidth=0.3)
        ax.legend(fontsize=9)
    else:
        scatter = ax.scatter(X_vis[:, 0], X_vis[:, 1], c=labels, cmap='viridis',
                             alpha=0.6, s=30, edgecolors='k', linewidth=0.3)
    
    n_clust = len(set(labels)) - (1 if -1 in labels else 0)
    ax.set_title(f'{name} — {n_clust} clusters')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

plt.suptitle('Clustering Algorithm Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Silhouette Score Comparison

Compute the silhouette score for each method. For DBSCAN, we exclude noise points from the calculation, so its score covers a different set of samples: report it with the noise fraction. Silhouette rewards compact, well-separated groups, so it favours K-Means-like partitions; it can be low or even negative for a density-connected region next to small dense pockets.

In [ ]:
# Compute silhouette scores
sil_km = silhouette_score(X_scaled, labels_km)
sil_agg = silhouette_score(X_scaled, labels_agg)
sil_gmm = silhouette_score(X_scaled, labels_gmm)

# DBSCAN: exclude noise points
non_noise = labels_db != -1
if non_noise.sum() > 1 and len(np.unique(labels_db[non_noise])) > 1:
    sil_db = silhouette_score(X_scaled[non_noise], labels_db[non_noise])
else:
    sil_db = np.nan

print("Silhouette Scores:")
print(f"  K-Means:      {sil_km:.4f}")
print(f"  DBSCAN:       {sil_db:.4f}  (noise excluded)")
print(f"  Hierarchical: {sil_agg:.4f}")
print(f"  GMM:          {sil_gmm:.4f}")

In [ ]:
# Bar chart of silhouette scores
method_names = ['K-Means', 'DBSCAN', 'Hierarchical', 'GMM']
sil_scores = [sil_km, sil_db, sil_agg, sil_gmm]
colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']

plt.figure(figsize=(8, 5))
bars = plt.bar(method_names, sil_scores, color=colors, edgecolor='black', linewidth=1.2)

for bar, score in zip(bars, sil_scores):
    if not np.isnan(score):
        plt.text(bar.get_x() + bar.get_width() / 2, max(score, 0) + 0.005,
                 f'{score:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.axhline(0, color='black', linewidth=0.8)
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score Comparison (DBSCAN: noise excluded)')
plt.ylim(min(0, np.nanmin(sil_scores)) - 0.05, np.nanmax(sil_scores) * 1.2)
plt.tight_layout()
plt.show()

---
## 9. Cluster Assignment Comparison

Use the Adjusted Rand Index (ARI) to measure pairwise agreement between the clustering methods. DBSCAN noise (−1) is treated as one more group here.

In [ ]:
# Pairwise ARI between methods
method_labels = {'K-Means': labels_km, 'DBSCAN': labels_db,
                 'Hierarchical': labels_agg, 'GMM': labels_gmm}

ari_matrix = pd.DataFrame(index=method_labels.keys(), columns=method_labels.keys(), dtype=float)

for m1, l1 in method_labels.items():
    for m2, l2 in method_labels.items():
        ari_matrix.loc[m1, m2] = adjusted_rand_score(l1, l2)

print("Adjusted Rand Index (pairwise):")
ari_matrix

In [ ]:
# Heatmap of pairwise ARI
plt.figure(figsize=(8, 6))
sns.heatmap(ari_matrix.astype(float), annot=True, fmt='.3f', cmap='YlGnBu',
            vmin=-0.1, vmax=1.0, linewidths=0.5)
plt.title('Pairwise Adjusted Rand Index Between Clustering Methods')
plt.tight_layout()
plt.show()

---
## 10. Summary Table and Discussion

In [ ]:
# Summary table
n_noise_db = (labels_db == -1).sum()
n_clusters_db_final = len(set(labels_db)) - (1 if -1 in labels_db else 0)

summary = pd.DataFrame({
    'Method': ['K-Means', 'DBSCAN', 'Hierarchical (Ward)', 'GMM'],
    'N Clusters': [3, n_clusters_db_final, 3, 3],
    'Noise Points': [0, n_noise_db, 0, 0],
    'Silhouette Score': [sil_km, sil_db, sil_agg, sil_gmm],
    'Cluster Type': ['Hard', 'Hard + Noise', 'Hard', 'Soft (probabilities)'],
    'Cluster Shape': ['Spherical', 'Arbitrary', 'Depends on linkage', 'Elliptical']
})

print("Clustering Algorithm Comparison Summary:")
summary

---
## Discussion: Which Method Works Best?

The "best" method depends on the goals and the data characteristics:

- **K-Means** is fast and works well when clusters are roughly spherical and of similar size. It requires specifying K upfront and is sensitive to outliers.

- **DBSCAN** is excellent at finding arbitrarily shaped clusters and automatically identifies noise/outliers. However, it requires careful tuning of `eps` and `min_samples`, and struggles with clusters of varying density.

- **Hierarchical Clustering** provides a full dendrogram that helps visualize the cluster hierarchy. The Ward method produces results similar to K-Means. It does not scale as well to very large datasets.

- **GMM** provides soft (probabilistic) assignments and can model elliptical clusters. BIC/AIC offer principled model selection. It assumes clusters follow Gaussian distributions.

For well log data, the choice depends on whether you expect:
- Well-separated, compact groups (K-Means or Ward)
- Noisy data with outliers (DBSCAN)
- Overlapping populations requiring probabilistic assignment (GMM)
- Need for a hierarchy of groupings (Hierarchical)

In practice, it is common to apply multiple methods and compare their results, as we have done here. Agreement between methods shows that a partition is robust to the choice of algorithm, not that the clusters are facies: a cluster label stays a hypothesis until core, cuttings or depth trends support it (compare with the depth plot from Lab 1).

---
## Summary

In this exercise we:

1. **Loaded and explored** a well log dataset and standardized the features.
2. **Applied four clustering algorithms** (K-Means, DBSCAN, Hierarchical, GMM) to the same dataset.
3. **Visualized all results** in a 2x2 comparison grid.
4. **Computed silhouette scores** to quantitatively compare cluster quality.
5. **Measured pairwise agreement** between methods using the Adjusted Rand Index.
6. **Created a summary table** capturing the key properties and results of each method.

**Key takeaways:**
- No single clustering algorithm is universally best — the choice depends on the data and the problem.
- Silhouette scores and ARI provide quantitative tools for comparison.
- Visual inspection remains essential for understanding cluster structure.
- DBSCAN is unique in its ability to detect noise and arbitrary-shape clusters.
- GMM is unique in providing probabilistic (soft) assignments.